# 02 - EDA y limpieza
**Maestría en IA - UDIT** · Lenguajes de Programación para IA · Prof. Carlos Cob
Alumno: Santiago Toledo Berral (STB)

Objetivo: ver qué explica el precio y dejar un dataset limpio en `data/train_clean.parquet`.
La columna importante es `catalog_content`: es un texto con campos (`Item Name`, `Bullet Point`, `Product Description`, `Value`, `Unit`) que hay que separar.

In [1]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

pio.renderers.default = "plotly_mimetype+notebook_connected"
pio.templates.default = "plotly_white"
AZUL, NARANJA, GRIS = "#2a78d6", "#eb6834", "#898781"
DATA_DIR = Path("../data")

## 1. Carga

In [2]:
df_raw = pd.read_csv(DATA_DIR / "train_full.csv")
print(df_raw.shape)
print("nulos:", df_raw.isna().sum().sum(),
      "| sample_id repetidos:", df_raw["sample_id"].duplicated().sum(),
      "| texto+precio repetidos:", df_raw.duplicated(["catalog_content", "price"]).sum(),
      "| imágenes repetidas:", df_raw["image_link"].duplicated().sum())
print(df_raw.loc[0, "catalog_content"][:600])

(75000, 4)
nulos: 0 | sample_id repetidos: 0 | texto+precio repetidos: 27 | imágenes repetidas: 2712
Item Name: La Victoria Green Taco Sauce Mild, 12 Ounce (Pack of 6)
Value: 72.0
Unit: Fl Oz



## 2. Separar los campos del texto

In [3]:
def extraer_campo(serie, campo):
    return serie.str.extract(rf"(?m)^{campo}:\s*(.*)$")[0].str.strip()

txt = df_raw["catalog_content"].fillna("")
df = df_raw.copy()
df["item_name"] = extraer_campo(txt, "Item Name")
df["description"] = txt.str.extract(r"(?ms)^Product Description:\s*(.*?)(?=^Value:|\Z)")[0].str.strip().fillna("")
df["n_bullets"] = txt.str.count(r"(?m)^Bullet Point")
df["unit_raw"] = extraer_campo(txt, "Unit")
df["value"] = pd.to_numeric(extraer_campo(txt, "Value"), errors="coerce")

# % de fichas que traen cada campo
pd.Series({"Item Name": df["item_name"].notna().mean(), "Bullet Points": (df["n_bullets"] > 0).mean(),
           "Description": (df["description"] != "").mean(), "Value > 0": (df["value"] > 0).mean()}).mul(100).round(1)

Item Name        100.0
Bullet Points     81.0
Description       43.4
Value > 0         98.7
dtype: float64

Nombre, cantidad y unidad están en casi todas las fichas. La descripción falta en más de la mitad, así que no me apoyo en ella.

## 3. Limpieza
La unidad aparece escrita de muchas formas (`oz`, `Ounces`, `fl oz`, `ct`, `gramm`...). La agrupo en tres familias y paso todo a gramos, mililitros o unidades.

In [4]:
def normalizar_unidad(u):
    """Devuelve (familia, factor para pasar a g / ml / unidades)."""
    if not isinstance(u, str):
        return "Desconocido", np.nan
    u = re.sub(r"\s+", " ", u.strip().lower().replace("(s)", "").replace(".", " ")).strip()
    if u in {"", "none", "-", "---"} or re.fullmatch(r"[\d ,]+", u):
        return "Desconocido", np.nan
    if "fl" in u.split()[0] or "fluid" in u:
        return "Volumen", 29.5735
    if u in {"ml", "millilitre", "milliliter", "mililitro", "millilitro"}:
        return "Volumen", 1.0
    if u in {"l", "liter", "liters", "litre", "litres", "ltr"}:
        return "Volumen", 1000.0
    if u in {"ounce", "ounces", "oz", "onzas"} or re.fullmatch(r"[\d,]+ oz", u):
        return "Peso", 28.3495
    if u in {"pound", "pounds", "lb", "lbs"}:
        return "Peso", 453.592
    if u in {"gram", "grams", "gramm", "gr", "g", "gm", "grams(gm)", "grammes"}:
        return "Peso", 1.0
    if u in {"kg", "kilogram", "kilograms"}:
        return "Peso", 1000.0
    conteo = ["count", "ct", "each", "piece", "unit", "pack", "bottle", "can", "jar", "bag", "box", "pouch",
              "carton", "k-cup", "capsule", "tea bag", "bucket", "liner", "unità", "per "]
    if any(c in u for c in conteo):
        return "Conteo", 1.0
    return "Otro", np.nan

mapa = {u: normalizar_unidad(u) for u in df["unit_raw"].dropna().unique()}
df["unit_family"] = df["unit_raw"].map(lambda u: mapa.get(u, ("Desconocido", np.nan))[0])
df["unit_factor"] = df["unit_raw"].map(lambda u: mapa.get(u, ("Desconocido", np.nan))[1])
print("Formas distintas de escribir la unidad:", df["unit_raw"].str.lower().nunique())
df["unit_family"].value_counts()

Formas distintas de escribir la unidad: 64


unit_family
Peso           44326
Conteo         18361
Volumen        11361
Desconocido      947
Otro               5
Name: count, dtype: int64

In [5]:
# Cantidad estándar (g / ml / unidades). Las cantidades 0 o sin unidad quedan como faltantes
df["qty_std"] = df["value"] * df["unit_factor"]
df.loc[df["qty_std"] <= 0, "qty_std"] = np.nan
df["qty_missing"] = df["qty_std"].isna().astype(int)
p999 = df.groupby("unit_family")["qty_std"].transform(lambda s: s.quantile(0.999))
df["qty_extreme"] = (df["qty_std"] > p999).astype(int)   # se marcan, no se borran

# Duplicados exactos fuera; grupos por texto e imagen para separar train/test sin fuga
df = df.drop_duplicates(["catalog_content", "price"]).reset_index(drop=True)
df["text_group"] = df.groupby("catalog_content", sort=False).ngroup()
df["image_group"] = df.groupby("image_link", sort=False).ngroup()
print(f"Filas: {len(df):,} | sin cantidad útil: {df['qty_missing'].mean():.1%} | cantidades extremas: {df['qty_extreme'].sum()}")

Filas: 74,973 | sin cantidad útil: 1.3% | cantidades extremas: 69


Variables nuevas a partir del nombre: unidades del pack (`Pack of 12`, `6-Pack`...), una categoría aproximada por palabras clave y la marca (primera palabra del nombre). El precio por kg / litro / unidad lo uso solo para analizar; no puede ser variable del modelo porque contiene el precio.

In [6]:
PATRONES_PACK = [r"(?:pack|case|box|set|bundle) of (\d{1,3})", r"(\d{1,3})\s*-?\s*(?:pack|pk|packs)\b",
                 r"(\d{1,3})\s*(?:per|/)\s*(?:case|pack|box|carton)",
                 r"\(\s*(\d{1,3})\s*(?:count|ct|pack|pk|bottles|cans|bags|boxes)\s*\)"]

def extraer_pack(nombre):
    n = nombre.lower() if isinstance(nombre, str) else ""
    for p in PATRONES_PACK:
        m = re.search(p, n)
        if m and 1 < int(m.group(1)) <= 200:
            return int(m.group(1))
    return 1

CATEGORIAS = [
    ("Café y té", r"coffee|tea\b|teas\b|k-cup|kcup|espresso|matcha|chai|cappuccino|latte"),
    ("Bebidas", r"juice|soda|\bwater\b|drink|beverage|wine|beer|kombucha|lemonade|sparkling|cola|smoothie"),
    ("Dulces y chocolate", r"candy|chocolate|gummy|gummies|gummi|caramel|lollipop|marshmallow|licorice|mint|candies"),
    ("Snacks", r"chip|cracker|snack|popcorn|pretzel|jerky|bar\b|bars\b|cookie|nuts?\b|almond|cashew|pistachio|trail mix"),
    ("Salsas y condimentos", r"sauce|seasoning|spice|salt|pepper|dressing|ketchup|mustard|vinegar|marinade|salsa|rub\b|extract|syrup|honey|oil\b"),
    ("Panadería, cereales y pasta", r"bread|cereal|oat|flour|granola|pasta|rice|noodle|baking|cake|mix\b|tortilla|quinoa|muffin|pancake"),
    ("Lácteos y proteína", r"cheese|milk|protein|butter|yogurt|creamer|meat|tuna|chicken|beef|salmon|egg"),
    ("Conservas y sopas", r"soup|broth|canned|beans|bean\b|tomato|pickle|olive|stock\b"),
    ("Frutas, verduras y semillas", r"fruit|berry|berries|apple|mango|vegetable|seed|raisin|date\b|fig\b|coconut"),
]

def categorizar(nombre):
    n = nombre.lower() if isinstance(nombre, str) else ""
    return next((cat for cat, patron in CATEGORIAS if re.search(patron, n)), "Otros")

def marca(nombre):
    palabras = re.sub(r"[^\w'&\- ]", " ", nombre).split() if isinstance(nombre, str) else []
    return palabras[0].title() if palabras else "Desconocida"

df["pack_count"] = df["item_name"].map(extraer_pack)
df["is_pack"] = (df["pack_count"] > 1).astype(int)
df["category"] = df["item_name"].map(categorizar)
df["brand"] = df["item_name"].map(marca)

df["has_description"] = (df["description"] != "").astype(int)
df["n_chars"] = df["catalog_content"].str.len()
df["n_words"] = df["catalog_content"].str.split().str.len()
df["n_digits"] = df["catalog_content"].str.count(r"\d")
df["upper_ratio"] = df["catalog_content"].str.count(r"[A-Z]") / df["n_chars"].clip(lower=1)
df["name_len"] = df["item_name"].fillna("").str.split().str.len()

df["log_price"] = np.log1p(df["price"])
df["price_per_unit"] = df["price"] / df["qty_std"] * np.where(df["unit_family"].isin(["Peso", "Volumen"]), 1000, 1)
df["category"].value_counts()

category
Café y té                      12763
Salsas y condimentos           11779
Otros                          10865
Bebidas                        10782
Snacks                         10362
Panadería, cereales y pasta     6865
Dulces y chocolate              4743
Lácteos y proteína              2593
Frutas, verduras y semillas     2248
Conservas y sopas               1973
Name: count, dtype: int64

## 4. El precio

In [7]:
print(df["price"].describe(percentiles=[0.01, 0.5, 0.99]).round(2).to_dict())
print(f"Asimetría: {df['price'].skew():.1f} -> con log1p: {df['log_price'].skew():.2f}")

p99 = df["price"].quantile(0.99)
fig = make_subplots(1, 2, subplot_titles=(f"Precio (hasta P99 = ${p99:.0f})", "Precio en escala log"))
for col, valores in [(1, df.loc[df["price"] <= p99, "price"]), (2, df["log_price"])]:
    cuentas, bordes = np.histogram(valores, bins=70)
    fig.add_trace(go.Bar(x=(bordes[:-1] + bordes[1:]) / 2, y=cuentas, marker_color=AZUL), 1, col)
marcas = [1, 3, 10, 30, 100, 300, 1000]
fig.update_xaxes(title_text="Precio ($)", row=1, col=1)
fig.update_xaxes(title_text="Precio ($, escala log)", tickvals=np.log1p(marcas), ticktext=[f"${m}" for m in marcas], row=1, col=2)
fig.update_layout(title="Distribución del precio", showlegend=False, bargap=0.05)
fig.show()

{'count': 74973.0, 'mean': 23.65, 'std': 33.38, 'min': 0.13, '1%': 1.32, '50%': 14.0, '99%': 145.26, 'max': 2796.0}
Asimetría: 13.6 -> con log1p: 0.20


In [8]:
# Outliers con IQR sobre el log del precio
q1, q3 = df["log_price"].quantile([0.25, 0.75])
lim_inf, lim_sup = q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1)
df["price_outlier"] = ((df["log_price"] < lim_inf) | (df["log_price"] > lim_sup)).astype(int)
print(f"Outliers: {df['price_outlier'].sum()} (fuera de ${np.expm1(lim_inf):.2f} - ${np.expm1(lim_sup):.2f})")
df.nlargest(5, "price")[["item_name", "price", "pack_count"]]

Outliers: 244 (fuera de $0.05 - $218.28)


,item_name,price,pack_count
58600,4Patriots 1-Year Survival Food Kit: Emergency ...,2796.00,1
32147,Royal Amber Osetra Caviar - Russian Sturgeon R...,1280.00,1
22836,495 days proteins meal 33 bottles x180 food 25...,1188.00,1
46290,"China Beauty Rings Tea (Loose) (8 oz, ZIN: 510...",1010.54,2
68364,"Organic Pistachios Shelled Raw, 30 Lbs. Bulk C...",921.50,1


La mayoría de productos cuesta entre $5 y $30, pero hay una cola larga hasta casi $2.800. Los precios más altos son reales (cajas, packs grandes, productos gourmet), así que no los elimino. Entreno con `log(1 + precio)` para que esos productos no dominen el error, y evalúo con SMAPE, que mide el error en porcentaje.

## 5. Qué explica el precio

In [9]:
# Precio por kg según categoría (solo productos vendidos por peso)
peso = df[(df["unit_family"] == "Peso") & df["price_per_unit"].notna()]
res = peso.groupby("category")["price_per_unit"].quantile([0.25, 0.5, 0.75]).unstack()
res.columns = ["p25", "mediana", "p75"]
res = res.sort_values("mediana")

fig = go.Figure(go.Scatter(x=res["mediana"], y=res.index, mode="markers", marker=dict(color=AZUL, size=11),
                           error_x=dict(type="data", symmetric=False, array=res["p75"] - res["mediana"],
                                        arrayminus=res["mediana"] - res["p25"], color=GRIS)))
fig.update_layout(title="Precio por kg según categoría (punto = mediana, línea = P25-P75)",
                  xaxis=dict(title="Precio por kg ($, escala log)", type="log"), height=450)
fig.show()
res.round(1)

,p25,mediana,p75
category,,,
Conservas y sopas,2.8,8.4,25.6
"Panadería, cereales y pasta",5.3,12.9,27.6
Lácteos y proteína,5.2,15.2,33.9
"Frutas, verduras y semillas",10.6,25.5,51.0
Snacks,13.0,26.6,50.3
Otros,10.5,27.1,70.0
Dulces y chocolate,15.2,27.5,50.3
Salsas y condimentos,11.3,30.5,70.4
Bebidas,12.8,31.0,64.5


In [10]:
# Descuento por volumen (productos vendidos por peso)
TRAMOS = ["< 100 g", "100-250 g", "250-500 g", "0,5-1 kg", "1-2 kg", "2-5 kg", "> 5 kg"]
peso = peso.assign(tramo=pd.cut(peso["qty_std"], [0, 100, 250, 500, 1000, 2000, 5000, np.inf], labels=TRAMOS))
vol = peso.groupby("tramo", observed=True).agg(precio=("price", "median"), precio_kg=("price_per_unit", "median"))

fig = make_subplots(1, 2, subplot_titles=("Precio del producto (mediana)", "Precio por kg (mediana)"))
fig.add_trace(go.Bar(x=vol.index.astype(str), y=vol["precio"], marker_color=AZUL, text=vol["precio"].round(2), textposition="outside"), 1, 1)
fig.add_trace(go.Bar(x=vol.index.astype(str), y=vol["precio_kg"], marker_color=NARANJA, text=vol["precio_kg"].round(1), textposition="outside"), 1, 2)
fig.update_yaxes(type="log", row=1, col=2)
fig.update_layout(title="A más cantidad, el producto es más caro pero el kilo más barato", showlegend=False, height=450)
fig.show()

- El tipo de producto marca el nivel de precio: el kilo va de unos $8 en conservas a $71 en café y té.
- La cantidad sube el precio, pero no de forma proporcional: de menos de 100 g a más de 5 kg el precio se multiplica por 2,6 y el precio por kg baja de $175 a $2,75. Por eso el modelo usa el log de la cantidad junto con el tipo de producto.
- `Value` ya es la cantidad total del pack, no hay que multiplicarla por el número de unidades.

In [11]:
orden_pack = ["1", "2", "3-4", "5-6", "7-12", "13-24", "> 24"]
df["pack_bucket"] = pd.cut(df["pack_count"], [0, 1, 2, 4, 6, 12, 24, np.inf], labels=orden_pack)
pack = df.groupby("pack_bucket", observed=True)["price"].agg(["size", "median"])
fig = px.bar(x=pack.index.astype(str), y=pack["median"], labels={"x": "Unidades en el pack", "y": "Precio mediano ($)"},
             title="Precio mediano según el tamaño del pack", color_discrete_sequence=[AZUL])
fig.show()

In [12]:
orden_cat = df.groupby("category")["price"].median().sort_values().index.tolist()
fig = px.box(df.sample(15_000, random_state=42), x="price", y="category", log_x=True, points=False,
             category_orders={"category": orden_cat}, color_discrete_sequence=[AZUL],
             labels={"price": "Precio ($, log)", "category": ""}, title="Precio por categoría", height=500)
fig.show()

frec = df["brand"].value_counts()
print(f"Marcas: {len(frec):,} | con un solo producto: {(frec == 1).mean():.0%} | top 500 cubren {frec.head(500).sum() / len(df):.0%} de los productos")

Marcas: 8,806 | con un solo producto: 42% | top 500 cubren 54% de los productos


- Los packs no ordenan el precio por sí solos: los de 3 a 12 unidades salen más baratos que los individuales porque suelen ser productos pequeños (latas, sobres).
- Dentro de cada categoría la dispersión es muy grande: la categoría orienta, pero no basta.
- Hay demasiadas marcas para usarlas directamente. En el notebook 04 se transforman en "precio típico de la marca", calculado solo con train.

## 6. Exportar

In [13]:
df["log_qty"] = np.log1p(df["qty_std"])
df["image_shared"] = (df.groupby("image_group")["price"].transform("size") > 1).astype(int)
cols = ["sample_id", "catalog_content", "image_link", "price", "log_price", "item_name", "description", "n_bullets",
        "has_description", "value", "unit_raw", "unit_family", "qty_std", "log_qty", "qty_missing", "qty_extreme",
        "pack_count", "is_pack", "category", "brand", "n_chars", "n_words", "n_digits", "upper_ratio", "name_len",
        "image_shared", "image_group", "text_group", "price_outlier"]
df[cols].to_parquet(DATA_DIR / "train_clean.parquet", index=False)
print(df[cols].shape)

(74973, 29)


## Conclusiones
- Los datos no tienen nulos, pero la información útil está dentro del texto. Quité 27 duplicados exactos.
- La unidad se escribía de 64 formas. Quedó en Peso, Volumen y Conteo, pasado a g, ml y unidades.
- El precio tiene una cola larga, así que el modelo trabaja con `log(1 + precio)` y se evalúa con SMAPE.
- Lo que más explica el precio es qué producto es (categoría, marca, texto del nombre) y cuánto trae, con descuento por volumen.
- Muchas imágenes se repiten entre variantes del mismo producto, así que separo train y test agrupando por imagen.